# Pipeline Completo: Transcrição de Reunião

Este notebook executa o pipeline end-to-end:

```
Arquivo de áudio → Transcrição → Diarização → Alinhamento → .md + .pdf
```

## Pré-requisitos

1. Configure um arquivo `.env` na raiz do projeto com `HF_TOKEN=hf_xxx`
2. Coloque o arquivo de áudio em `data/audio/`
3. Execute as células em ordem

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")

from src.config import build_settings
cfg = build_settings()

print("Configurações carregadas:")
print(f"  Whisper model : {cfg.whisper_model}")
print(f"  Idioma        : {cfg.language}")
print(f"  HF_TOKEN      : {'configurado' if cfg.hf_token else 'NÃO CONFIGURADO ⚠️'}")
print(f"  Audio dir     : {cfg.audio_dir}")
print(f"  Output dir    : {cfg.output_dir}")

In [ ]:
# Listar arquivos de áudio disponíveis
audio_files = list(cfg.audio_dir.glob("*"))
if audio_files:
    print("Arquivos disponíveis:")
    for f in audio_files:
        print(f"  - {f.name}")
else:
    print("⚠️  Nenhum arquivo encontrado em", cfg.audio_dir)
    print("   Coloque um arquivo de áudio em data/audio/ e execute novamente.")

In [ ]:
# Defina o arquivo a transcrever
AUDIO_FILE = cfg.audio_dir / "reuniao.mp3"  # <- Ajuste o nome

assert AUDIO_FILE.exists(), f"Arquivo não encontrado: {AUDIO_FILE}"
print("Arquivo selecionado:", AUDIO_FILE)

## Executar o Pipeline

In [ ]:
from src.pipeline import run_pipeline

result = run_pipeline(audio_path=AUDIO_FILE, config=cfg)

segments = result["segments"]
md_path  = result["md"]
pdf_path = result["pdf"]

print(f"\nPipeline concluído!")
print(f"  Segmentos      : {len(segments)}")
print(f"  Markdown gerado: {md_path}")
print(f"  PDF gerado     : {pdf_path}")

## Visualizar Resultado

In [ ]:
from collections import Counter

speaker_counts = Counter(seg.speaker for seg in segments)
print("Speakers identificados:")
for speaker, count in sorted(speaker_counts.items()):
    print(f"  {speaker}: {count} segmentos")

In [ ]:
print("Primeiros 10 segmentos:\n")
for seg in segments[:10]:
    h_s, m_s, s_s = int(seg.start)//3600, (int(seg.start)%3600)//60, int(seg.start)%60
    h_e, m_e, s_e = int(seg.end)//3600, (int(seg.end)%3600)//60, int(seg.end)%60
    print(f"[{h_s:02d}:{m_s:02d}:{s_s:02d} → {h_e:02d}:{m_e:02d}:{s_e:02d}] {seg.speaker}: {seg.text}")

In [ ]:
# Visualizar o conteúdo do Markdown gerado
print(md_path.read_text(encoding="utf-8")[:2000])